# Script Outline

Import ACS 5 Year Estimates table and import census tracts and counties layers from TIGER.  Merge these files together to make two maps, a map of census tracts and a map of counties.

- Prepare Workspace
- Import Data
- Data Cleaning
- Maps
- Exports

CRS Reprojection source
https://spatialreference.org/ref/epsg/2226/


## Prepare Workspace

#### Import packages

In [ ]:
# General
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import json


# Geographic
import geopandas as gpd
from census import Census
from us import states
import censusdata as acs
import pyproj

# Plotting
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

#### File paths

In [ ]:
# Working directory
path_projects = os.path.dirname(os.path.dirname(os.getcwd()))
print(path_projects)

# Set file paths
rootpath = os.path.join(path_projects, 'SACOG Exam')
path_in     = os.path.join(rootpath, 'Raw Data'     )
path_out    = os.path.join(rootpath, 'Python Output')
path_config = os.path.join(rootpath, 'config'       )

In [ ]:
# Supress scientific notation
pd.options.display.float_format = '{:.2f}'.format

## Import Data

In [ ]:
# Import county FIPS mapping
df_fips = pd.read_excel(os.path.join(path_out, 'County to FIPS Code Mapping.xlsx')
                        , dtype={'State FIPS': object, 'County FIPS': object})
df_fips.head()

In [ ]:
# Import ACS5 data
df_acs5 = pd.read_excel(os.path.join(path_out, 'Step 01b_Cleaned ACS5 Table B08006_2024-02-24.xlsx')
                      , dtype={'State FIPS': object, 'County FIPS': object, 'Tract ID': object})
df_acs5.head(3)

In [ ]:
# Access shapefile of California Census Tracts
# initialize empty list to store geopandas dataframes
# set which county FIPS, and years to import
# import using TIGER url query
# set year
# concatenate

list_gpd_tracts = []
years = [2018, 2019, 2020, 2021]
for year in tqdm(years):
    temp = gpd.read_file("https://www2.census.gov/geo/tiger/TIGER" + str(year) + "/TRACT/tl_" + str(year) + "_06_tract.zip")
    temp['Year'] = year
    list_gpd_tracts.append(temp)
gpd_tracts = pd.concat(list_gpd_tracts)    


# Reproject shapefile to UTM Zone 17N
# Print GeoDataFrame of shapefile
# Check shapefile projection

gpd_tracts = gpd_tracts.to_crs(epsg = 2226)
print(gpd_tracts.head(2))
print('Shape: ', gpd_tracts.shape)
print("\nThe shapefile projection is: {}".format(gpd_tracts.crs))

## Data Cleaning

In [ ]:
# Combine state, county, and tract columns together to create a new string and assign to new column
df_acs5["GEOID"] = df_acs5["State FIPS"] + df_acs5["County FIPS"] + df_acs5["Tract ID"]
df_acs5_merge = df_acs5.drop(columns = ["State FIPS", "County FIPS", "Tract ID"])
df_acs5_merge.head(3)

In [ ]:
# Check merge types
print(type(df_acs5_merge['GEOID'].values[0]))
print(type(gpd_tracts   ['GEOID'].values[0]))

In [ ]:
# Join the attributes of the dataframes together
# Source: https://geopandas.org/docs/user_guide/mergingdata.html
gpd_tracts_acs = gpd_tracts.merge(df_acs5_merge, on = ["GEOID", "Year"])
gpd_tracts_acs = gpd_tracts_acs[["GEOID", "COUNTYFP", "geometry", "Year", 'County Name', 'Census Tract Name'
                                 , "Population"
                                 , "Population commuting alone"
                                 , "Population working from home"]]

# Show
print(gpd_tracts_acs.head(2))
print('Shape: ', gpd_tracts_acs.shape)

## Maps

#### Census Tracts

In [ ]:
# Estimate percentages
gpd_tracts_acs["Population commuting alone (%)"  ] = gpd_tracts_acs["Population commuting alone"  ]/gpd_tracts_acs["Population"]
gpd_tracts_acs["Population working from home (%)"] = gpd_tracts_acs["Population working from home"]/gpd_tracts_acs["Population"]

# Normalize metrics by population
gpd_tracts_acs["Population commuting alone_per 1k pop"  ] = gpd_tracts_acs["Population commuting alone"  ]/(gpd_tracts_acs['Population']/1000)
gpd_tracts_acs["Population working from home_per 1k pop"] = gpd_tracts_acs["Population working from home"]/(gpd_tracts_acs['Population']/1000)

# Normalize metrics by land area
gpd_tracts_acs['acres'] = gpd_tracts_acs['geometry'].area / 43560 # acres of each county
gpd_tracts_acs["Population commuting alone_per 1k acres"  ] = gpd_tracts_acs["Population commuting alone"  ]/(gpd_tracts_acs['acres']/1000)
gpd_tracts_acs["Population working from home_per 1k acres"] = gpd_tracts_acs["Population working from home"]/(gpd_tracts_acs['acres']/1000)

In [ ]:
# set metric to map
# Create subplots and plot data

metric = "Population commuting alone_per 1k acres"

fig, ax = plt.subplots(1, 1, figsize = (20, 10))

gpd_tracts_acs.plot(column = metric,
                       ax = ax,
                       cmap = "RdPu",
                       legend = True)

plt.style.use('bmh')
ax.set_title(metric, fontdict = {'fontsize': '25', 'fontweight' : '3'})

In [ ]:
# # Organize specific dataframe and geojson object for mapping with px.choropleth
# # Create index field for mapping
# # Convert to 4326 CRS for mapping


# temp = gpd_tracts_acs.copy()
# temp.reset_index(inplace = True)
# temp.rename(columns = {'index':'ID'}, inplace = True)

# map_df = temp[['ID', 'geometry']]
# map_df.to_crs(pyproj.CRS.from_epsg(4326), inplace=True)

# df = temp.drop(columns = ['geometry'])

# # join the geodataframe with the cleaned up csv dataframe
# merged = map_df.set_index('ID').join(df.set_index('ID'))

In [ ]:
# # Iterate through every metric for mapping
# # Export maps as interactive plotly maps in .html files

# metrics = ["Population"
#           , "Population commuting alone"  
#           , "Population working from home"
#           , "Population commuting alone (%)"  
#           , "Population working from home (%)"
#           , "Population commuting alone_per 1k pop"  
#           , "Population working from home_per 1k pop"
#           , "Population commuting alone_per 1k acres"  
#           , "Population working from home_per 1k acres"]


# for metric in tqdm(metrics):
#     fig = px.choropleth(merged
#                         , geojson=merged.geometry
#                         , locations=merged.index
#                         , color=metric
#                         , color_continuous_scale="Blues"
#                         , animation_frame = 'Year')
#     fig.update_geos(fitbounds="locations", visible=False)
    
#     fig.write_html(
#         os.path.join(
#             path_out
#             , 'plotly'
#             , 'ACS5'
#             , 'maps'
#             , ''.join([metric
#                        , "_Tracts_"
#                        , '_ACS5_'
#                        , date.today().strftime("%Y-%m-%d")
#                        , '.html'])
#         )
#     )
    

#### Counties

In [ ]:
# Dissolve and group the census tracts within each county and aggregate all the values together
# Source: https://geopandas.org/docs/user_guide/aggregation_with_dissolve.html
gpd_counties_acs = gpd_tracts_acs.drop(columns = ["GEOID", "County Name", "Census Tract Name"])\
                            .dissolve(by = ['COUNTYFP', 'Year'], aggfunc = 'sum', as_index = False)

# Show dataframe
print(gpd_counties_acs.head(2))
print('Shape: ', gpd_counties_acs.shape)

In [ ]:
# Estimate percentages
gpd_counties_acs["Population commuting alone (%)"  ] = gpd_counties_acs["Population commuting alone"  ]/gpd_counties_acs["Population"]
gpd_counties_acs["Population working from home (%)"] = gpd_counties_acs["Population working from home"]/gpd_counties_acs["Population"]

# Normalize metrics by population
gpd_counties_acs["Population commuting alone_per 1k pop"  ] = gpd_counties_acs["Population commuting alone"  ]/(gpd_counties_acs['Population']/1000)
gpd_counties_acs["Population working from home_per 1k pop"] = gpd_counties_acs["Population working from home"]/(gpd_counties_acs['Population']/1000)

# Normalize metrics by land area
gpd_counties_acs['acres'] = gpd_counties_acs['geometry'].area / 43560 # acres of each county
gpd_counties_acs["Population commuting alone_per 1k acres"  ] = gpd_counties_acs["Population commuting alone"  ]/(gpd_counties_acs['acres']/1000)
gpd_counties_acs["Population working from home_per 1k acres"] = gpd_counties_acs["Population working from home"]/(gpd_counties_acs['acres']/1000)

In [ ]:
# set metric to map
# Create subplots and plot data


metric = "Population working from home_per 1k pop"

fig, ax = plt.subplots(1, 1, figsize = (20, 10))

gpd_counties_acs.plot(column = metric,
                       ax = ax,
                       cmap = "RdPu",
                       legend = True)

plt.style.use('bmh')
ax.set_title(metric, fontdict = {'fontsize': '25', 'fontweight' : '3'})

In [ ]:
# # # Organize specific dataframe and geojson object for mapping with px.choropleth
# # # Create index field for mapping
# # # Convert to 4326 CRS for mapping


# temp = gpd_counties_acs.copy()
# temp.reset_index(inplace = True)
# temp.rename(columns = {'index':'ID'}, inplace = True)

# map_df = temp[['ID', 'geometry']]
# map_df.to_crs(pyproj.CRS.from_epsg(4326), inplace=True)

# df = temp.drop(columns = ['geometry'])

# # join the geodataframe with the cleaned up csv dataframe
# merged = map_df.set_index('ID').join(df.set_index('ID'))

In [ ]:
# # Iterate through every metric for mapping
# # Export maps as interactive plotly maps in .html files

# metrics = ["Population"
#           , "Population commuting alone"  
#           , "Population working from home"
#           , "Population commuting alone (%)"  
#           , "Population working from home (%)"
#           , "Population commuting alone_per 1k pop"  
#           , "Population working from home_per 1k pop"
#           , "Population commuting alone_per 1k acres"  
#           , "Population working from home_per 1k acres"]


# for metric in tqdm(metrics):
#     fig = px.choropleth(merged
#                         , geojson=merged.geometry
#                         , locations=merged.index
#                         , color=metric
#                         , color_continuous_scale="Blues"
#                         , animation_frame = 'Year')
#     fig.update_geos(fitbounds="locations", visible=False)
    
#     fig.write_html(
#         os.path.join(
#             path_out
#             , 'plotly'
#             , 'ACS5'
#             , 'maps'
#             , ''.join([metric
#                        , "_Counties_"
#                        , '_ACS5_'
#                        , date.today().strftime("%Y-%m-%d")
#                        , '.html'])
#         )
#     )

## Exports

In [ ]:
# df_acs5_counties = gpd_counties_acs.drop(columns = 'geometry')
# df_acs5_tracts   = gpd_tracts_acs  .drop(columns = 'geometry')

In [ ]:
# # Set output name
# name_output_counties = ['Step 02b_ACS5 Transportation Metrics_', 'Counties_'  , date.today().strftime("%Y-%m-%d"),'.xlsx']
# name_output_tracts   = ['Step 02b_ACS5 Transportation Metrics_', 'Tracts_'    , date.today().strftime("%Y-%m-%d"),'.xlsx']
# name_output_counties = "".join(name_output_counties)
# name_output_tracts   = "".join(name_output_tracts  )

In [ ]:
# # Export to excel files
# df_acs5_counties.to_excel(os.path.join(path_out, name_output_counties), index=False)
# df_acs5_tracts  .to_excel(os.path.join(path_out, name_output_tracts  ), index=False)

In [ ]:
# # Export to geojson files
# gpd_counties_acs.to_file(os.path.join(path_out, "gpd_counties_acs5.geojson"), driver="GeoJSON")
# gpd_tracts_acs  .to_file(os.path.join(path_out, "gpd_tracts_acs5  .geojson"), driver="GeoJSON")